<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 5 (2): LangGraph — Drawing the Loop

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. Understand **state** — the dictionary that every node reads from and writes to
2. Build a graph from **nodes** and **edges**, and draw it
3. Let the graph **branch** with a conditional edge
4. Tell a **workflow** from an **agent**, and know which one you actually need
5. Put an **LLM** inside a node
6. Rebuild notebook 1's agent as a graph with **`ToolNode`** and **`tools_condition`**

> **Notebook 1 first.** This one assumes you know what a tool call is and have seen the
> agent loop written by hand. Here we rebuild that loop as something you can draw.

---

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q langgraph langchain-openai

In [ ]:
import os
from getpass import getpass
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
MODEL = "gpt-4o-mini"

print("Setup complete")

---

## 2. Why a Graph?

In notebook 1 you wrote this:

```python
for step in range(MAX_ITERATIONS):
    response = client.chat.completions.create(...)
    if response.choices[0].finish_reason != "tool_calls":
        return response.choices[0].message.content
    ...run the tools and go round again
```

It works. But everything about it is invisible: the data lives in local variables and the control
flow lives inside an `if`.

**LangGraph is that same loop, written as a picture.**

Three words carry the whole library:

```
  STATE   a dictionary that flows through the graph
  NODE    a function: reads the state, returns an update to it
  EDGE    what runs next
```

That's it. Let's build one with no LLM at all, so the mechanics are unmistakable.

---

## 3. Your First Graph

We'll build a support reply, one piece at a time. Three nodes, and **one field of state** that each
one adds to.

**Step 1 — describe the state.** This is the dictionary that travels through the graph. To keep it
simple it has exactly one field.

In [ ]:
class ReplyState(TypedDict):
    reply: str          # the ONE field every node will read and update


print(ReplyState.__annotations__)

**Step 2 — write the nodes.** A node is a plain function. There are only two things it does:

```
  READ    state["reply"]              <- get the current value
  UPDATE  return {"reply": new_value} <- return the field you changed
```

That's the whole contract. No LLM here on purpose — a node is just a function.

In [ ]:
def add_greeting(state: ReplyState):
    print("  [node] add_greeting")
    return {"reply": state["reply"] + "Hello! "}          # read it, add to it, return it


def add_answer(state: ReplyState):
    print("  [node] add_answer")
    return {"reply": state["reply"] + "Your order ships tomorrow. "}


def add_signature(state: ReplyState):
    print("  [node] add_signature")
    return {"reply": state["reply"] + "- The Support Team"}

**Step 3 — wire them together.** Add each node, then say what follows what. `START` and `END`
are the two built-in markers.

In [ ]:
builder = StateGraph(ReplyState)

# Nodes: a name, and the function to run
builder.add_node("add_greeting", add_greeting)
builder.add_node("add_answer", add_answer)
builder.add_node("add_signature", add_signature)

# Edges: what runs next
builder.add_edge(START, "add_greeting")
builder.add_edge("add_greeting", "add_answer")
builder.add_edge("add_answer", "add_signature")
builder.add_edge("add_signature", END)

graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

**Step 4 — run it.** `invoke()` takes the starting state and returns the final state.

In [ ]:
result = graph.invoke({"reply": ""})      # start with an empty string

print(result)

Look at what happened to that one field:

```
  ""                                                        <- what you passed in
  "Hello! "                                                 <- after add_greeting
  "Hello! Your order ships tomorrow. "                      <- after add_answer
  "Hello! Your order ships tomorrow. - The Support Team"    <- after add_signature
```

Each node **read** `state["reply"]`, **returned** a new value for it, and LangGraph carried that
forward to the next node.

> That is the whole idea of state: nodes don't call each other and don't share variables. They pass
> a dictionary along, and each one changes the part it cares about.

---

## 4. Conditional Edges — Letting the Graph Branch

So far the path was fixed: greeting, answer, signature, every single time.

A **conditional edge** lets the graph choose. You write a **router**: a function that looks at the
state and returns **the name of the next node**, as a plain string.

This graph has two fields — what the customer wrote, and what we send back.

In [ ]:
class TicketState(TypedDict):
    text: str       # what the customer wrote
    reply: str      # what we send back


def route_ticket(state: TicketState) -> str:
    """A router returns the NAME of the next node - just a string."""
    if "refund" in state["text"].lower():
        return "billing"
    else:
        return "general"

In [ ]:
def billing(state: TicketState):
    print("  [node] billing")
    return {"reply": "Our billing team will review your refund within 2 working days."}


def general(state: TicketState):
    print("  [node] general")
    return {"reply": "Thanks for your message - we will get back to you soon."}

In [ ]:
ticket_builder = StateGraph(TicketState)

ticket_builder.add_node("billing", billing)
ticket_builder.add_node("general", general)

# The conditional edge: instead of a fixed START -> node edge,
# ask the router which node should run
ticket_builder.add_conditional_edges(START, route_ticket)

ticket_builder.add_edge("billing", END)
ticket_builder.add_edge("general", END)

ticket_graph = ticket_builder.compile()

display(Image(ticket_graph.get_graph().draw_mermaid_png()))

The dotted lines in that picture are the branch. Change the message below and re-run — watch a
different path light up.

In [ ]:
# Change this line and re-run: try "I want a refund" / "where is my order"
message = "I want a refund for my last invoice"

result = ticket_graph.invoke({"text": message, "reply": ""})
print(result["reply"])

### You just built the routing pattern

That graph is a **workflow**: you decided the possible paths in advance, and a router picks one.
Compare it with notebook 1's agent, where the *model* decided what to do next.

| | **Workflow** *(this graph)* | **Agent** *(notebook 1)* |
|---|---|---|
| Who chooses the path | **you**, at write time | **the model**, at run time |
| Possible outcomes | you can list them all | you cannot |
| Testing | straightforward | hard - it varies |
| Cost | predictable | varies per question |

> **If you can write down the steps, write down the steps.** Autonomy is a cost, not a feature -
> every decision you hand to the model is one you can no longer test, price, or explain to a
> customer. Most production systems marketed as "AI agents" are workflows, and they are right to be.

Reach for an agent only when you genuinely **cannot** list the steps in advance.

---

## 5. Putting an LLM in a Node

A node is just a function, so a node that calls an LLM is just a function that calls an LLM.

One new thing: for conversations the state holds a **list of messages** that we want to **add to**
rather than overwrite. That is what `add_messages` does.

In [ ]:
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL)


class ChatState(TypedDict):
    # Annotated[..., add_messages] means "when a node returns messages, APPEND them"
    messages: Annotated[list, add_messages]


def chatbot(state: ChatState):
    """One LLM call, as a node."""
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
chat_builder = StateGraph(ChatState)
chat_builder.add_node("chatbot", chatbot)
chat_builder.add_edge(START, "chatbot")
chat_builder.add_edge("chatbot", END)

chat_graph = chat_builder.compile()

result = chat_graph.invoke({"messages": [{"role": "user", "content": "What is the capital of India?"}]})
print(result["messages"][-1].content)

> Without `add_messages`, the second node to return `messages` would **overwrite** the first.
> With it, the conversation accumulates. That one annotation is what makes chat work.

---

## 6. Tools in a Graph — the Loop, Drawn

Now we rebuild notebook 1's agent. Same idea, three things get shorter:

| Notebook 1 (by hand) | Here |
|---|---|
| you wrote the JSON schema | the **`@tool`** decorator builds it from your type hints and docstring |
| `handle_tool_calls()` | **`ToolNode`** |
| `if finish_reason == "tool_calls"` | **`tools_condition`** |

The docstring **is** the description the model reads - the same rule as notebook 1.

In [ ]:
from langchain_core.tools import tool
from datetime import datetime


@tool
def get_current_time() -> str:
    """Get the current date and time. Use for anything about 'today', 'now', or dates."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S (%A)")


@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression such as '250 * 50000'. Use for any calculation."""
    return str(eval(expression))


tools = [get_current_time, calculate]

# Look at what the decorator generated for you
print(calculate.name)
print(calculate.description)

In [ ]:
# bind_tools tells the model which tools exist - the same job the schemas did in notebook 1
llm_with_tools = ChatOpenAI(model=MODEL).bind_tools(tools)


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: AgentState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

agent_builder = StateGraph(AgentState)

agent_builder.add_node("agent", agent_node)
agent_builder.add_node("tools", ToolNode(tools=tools))     # runs whatever the model asked for

agent_builder.add_edge(START, "agent")

# tools_condition is a ready-made router: "tools" if the model asked for one, END if it didn't
agent_builder.add_conditional_edges("agent", tools_condition)

# ...and this edge is what makes it a LOOP - after the tools run, go back to the model
agent_builder.add_edge("tools", "agent")

agent_graph = agent_builder.compile()

display(Image(agent_graph.get_graph().draw_mermaid_png()))

**Look at that picture.** The arrow from `tools` back to `agent` is the cycle - and that cycle
**is** the `while` loop you wrote in notebook 1. Same behaviour, now something you can point at.

In [ ]:
# Needs a tool
result = agent_graph.invoke({"messages": [{"role": "user", "content": "What is 47853 * 1942?"}]})
print(result["messages"][-1].content)

In [ ]:
# Needs nothing - tools_condition sends it straight to END, so the loop never runs
result = agent_graph.invoke({"messages": [{"role": "user", "content": "What is the capital of France?"}]})
print(result["messages"][-1].content)

In [ ]:
# Every step of the run is in the message list - this is your loop trace, for free
result = agent_graph.invoke({"messages": [{"role": "user", "content": "What day is it today?"}]})

print(f"{len(result['messages'])} messages in the trace:")
print([type(m).__name__ for m in result["messages"]])

---

## 7. Exercises

Fill in the blanks (`___`). Hints are in the comments.

### Q1: Add a node to the first graph

Add a node that puts the ticket number at the front of the reply, and run it **first**.

In [ ]:
def add_ticket_number(state: ReplyState):
    # Hint: read the field with state["reply"], return the field you changed.
    return {"___": "[Ticket #4471] " + state["___"]}


b = StateGraph(ReplyState)
b.add_node("add_ticket_number", add_ticket_number)
b.add_node("add_greeting", add_greeting)
b.add_node("add_answer", add_answer)
b.add_node("add_signature", add_signature)

b.add_edge(START, "___")                      # which node runs first now?
b.add_edge("add_ticket_number", "add_greeting")
b.add_edge("add_greeting", "add_answer")
b.add_edge("add_answer", "add_signature")
b.add_edge("add_signature", END)

print(b.compile().invoke({"reply": ""}))

### Q2: Add a third branch

Route tickets containing "delivery" to a new `shipping` node.

In [ ]:
def shipping(state: TicketState):
    return {"reply": "___"}


def route_v2(state: TicketState) -> str:
    # Hint: a router just returns a node NAME as a string.
    if "refund" in state["text"].lower():
        return "billing"
    elif "___" in state["text"].lower():
        return "___"
    else:
        return "general"


b = StateGraph(TicketState)
b.add_node("billing", billing)
b.add_node("general", general)
b.add_node("shipping", ___)
b.add_conditional_edges(START, ___)
b.add_edge("billing", END)
b.add_edge("general", END)
b.add_edge("shipping", END)

print(b.compile().invoke({"text": "where is my delivery", "reply": ""})["reply"])

### Q3: Give the agent a tool of your own

Write a `@tool` the model cannot answer without, add it to the agent graph, and ask a question that
needs it.

In [ ]:
@tool
def get_employee_count() -> str:
    """___"""                       # remember: the docstring IS the description
    return "250"


my_tools = [get_current_time, calculate, ___]
my_llm = ChatOpenAI(model=MODEL).bind_tools(___)


def my_agent_node(state: AgentState):
    return {"messages": [my_llm.invoke(state["messages"])]}


b = StateGraph(AgentState)
b.add_node("agent", my_agent_node)
b.add_node("tools", ToolNode(tools=___))
b.add_edge(START, "agent")
b.add_conditional_edges("agent", tools_condition)
b.add_edge("tools", "agent")

r = b.compile().invoke({"messages": [{"role": "user", "content": "___"}]})
print(r["messages"][-1].content)

---

## Key Takeaways

1. **Three words: state, node, edge.** State is a dictionary that flows; a node is a function that
   reads it and returns an update; an edge says what runs next.

2. **A node does two things.** Read a field with `state["field"]`, and update it by returning
   `{"field": new_value}`. Nodes never call each other — they pass the state along.

3. **A conditional edge is a router** — a function that returns the *name* of the next node, as a
   string. That is the routing pattern, and it is the cheapest useful thing in this notebook.

4. **Workflow vs agent: who picks the path.** You, at write time, or the model, at run time. If you
   can write down the steps, write down the steps.

5. **`add_messages` means append, not overwrite.** Without it a conversation cannot accumulate.

6. **`ToolNode` + `tools_condition` + one edge back = the agent loop**, drawn as a cycle. Same
   behaviour as notebook 1, now visible.

### Concept Map

```
            START
              |
              v
        +-----------+   tools_condition says "tools"    +---------+
        |   agent   | --------------------------------> |  tools  |
        | (LLM node)| <-------------------------------- | ToolNode|
        +-----------+       results appended            +---------+
              |
              | tools_condition says END
              v
             END

   state ---> every node reads it, and returns only the field it changed
```

### Quick Reference

| Piece | What it does |
|---|---|
| `TypedDict` | describes the shape of the state |
| `state["field"]` | **read** a field inside a node |
| `return {"field": value}` | **update** that field |
| `Annotated[list, add_messages]` | append to this key instead of overwriting |
| `StateGraph(State)` | the builder |
| `add_node("name", fn)` | register a step |
| `add_edge(a, b)` | always go from a to b |
| `add_conditional_edges(a, router)` | ask the router which node comes next |
| `START` / `END` | the built-in entry and exit markers |
| `.compile()` | turn the builder into a runnable graph |
| `.invoke(state)` | run it; returns the final state |
| `draw_mermaid_png()` | draw it |
| `@tool` | build the schema from type hints + docstring |
| `bind_tools(tools)` | tell the model which tools exist |
| `ToolNode(tools)` | run whatever the model asked for |
| `tools_condition` | ready-made router: tools, or END |

> **There is more in LangGraph than we used.** A *checkpointer* lets a graph remember a conversation
> across runs and even **pause** before a step so a human can approve it. Worth knowing it exists —
> you don't need it for anything today.

### 🏠 Homework

1. **Draw before you build.** Sketch a graph for a task you care about — nodes, edges, and one
   branch — then implement it and compare your picture with `draw_mermaid_png()`.
2. **Break the loop.** In the agent graph, delete `add_edge("tools", "agent")` and re-run a question
   that needs a tool. What comes back, and why? That one edge was doing a lot of work.
3. **Workflow or agent?** Take three tasks from your own life and decide which each one needs. Write
   one sentence of justification for each.

### 📚 Resources

- [LangGraph documentation](https://langchain-ai.github.io/langgraph/)
- [LangGraph — graph API](https://langchain-ai.github.io/langgraph/concepts/low_level/)
- [Anthropic — Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)

---

**Next:** notebook 3 goes one level higher again — **CrewAI**, where you describe agents by their
*role* and let a crew divide the work.